# STA6130 Week 8 Group Assignment: Forecasting as Regression and Feature Engineering
**Dataset:** Store Item Demand Forecasting Challenge (Kaggle), daily unit sales, 1 Jan 2013 to 31 Dec 2017  
**Competition:** https://www.kaggle.com/c/demand-forecasting-kernels-only  (data: `train.csv` with columns `date, store, item, sales`)  

**Mapping to the assignment:** sections 1 to 10 below follow the ten required report items, so each output can be pasted straight into the matching part of the report.

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 1. Describe the forecasting problem

In [2]:
def find_train_csv():
    # 1) a copy next to the notebook, 2) the Kaggle notebook input folder, 3) kagglehub download
    for path in ["train.csv", "/kaggle/input/demand-forecasting-kernels-only/train.csv"]:
        if os.path.exists(path):
            return path
    try:
        import kagglehub
        folder = kagglehub.competition_download("demand-forecasting-kernels-only")
        hits = glob.glob(os.path.join(folder, "**", "train.csv"), recursive=True)
        if hits:
            return hits[0]
    except Exception as e:
        print("kagglehub download failed:", e)
    raise FileNotFoundError(
        "train.csv not found. Join the competition, download the data from"
        )

raw = pd.read_csv(find_train_csv(), parse_dates=["date"])
print("Full file:", raw.shape, "| stores:", raw["store"].nunique(), "| items:", raw["item"].nunique())
print("Period:", raw["date"].min().date(), "to", raw["date"].max().date())

# One store-item series (local model); change these IDs to select a different series.
STORE = 1
ITEM = 1
s = (raw[(raw["store"] == STORE) & (raw["item"] == ITEM)]
     .sort_values("date").set_index("date")["sales"].astype(float).asfreq("D"))
assert s.isna().sum() == 0, "missing days found: impute carefully (never with future values)"
print(f"\nSeries: store {STORE}, item {ITEM}: {len(s)} daily observations, {s.index.min().date()} to {s.index.max().date()}")
print(s.describe().round(2))

Full file: (913000, 4) | stores: 10 | items: 50
Period: 2013-01-01 to 2017-12-31

Series: store 1, item 1: 1826 daily observations, 2013-01-01 to 2017-12-31
count    1826.00
mean       19.97
std         6.74
min         4.00
25%        15.00
50%        19.00
75%        24.00
max        50.00
Name: sales, dtype: float64
